# Neural SINDy: Sparse System Identification with Grokked MLP Libraries
- **GPU**: NVIDIA A100 with 80GB High RAM
- **Python**: 3.10 or higher
- **CUDA**: 12.1 or higher
- **PyTorch**: 2.1 or higher
- Paper: https://huggingface.co/pandevim/cs810/tree/main/paper
- Google Colab (experiment code): https://colab.research.google.com/drive/1BI5pjA5e4O-FTY6DUd1JVqxQdXxY5EHP?usp=sharing
- Hugging Face (intermediate weights and checkpoints): https://huggingface.c (o/pandevim/cs810/tree/main
- W&B Dashboard (experiment observations): https://wandb.ai/pandevim-wandb-team/cs810/reports/Neural-SINDy-Experiment-Results--VmlldzoxNjI5NjAwOA


In [1]:
%%capture
!pip install pysr

In [25]:
import io
import os
import subprocess

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import json
from tqdm.auto import tqdm

from sklearn.linear_model import Ridge

from google.colab import userdata
from huggingface_hub import HfApi, login, hf_hub_download
import wandb

In [ ]:
%%capture
wandb.login("REDACTED_WANDB_TOKEN")
os.environ["HF_TOKEN"] = "REDACTED_HF_TOKEN"

In [30]:
api = HfApi()
HF_REPO_ID = "pandevim/cs810"

In [31]:
device = "cuda"
torch.set_default_device(device)

In [ ]:
""" Clones a git repository or pulls updates if it already exists """
repo_url, base_path = "https://github.com/pandevim/NeuralSINDy.git", "/content"
os.chdir(base_path)
repo_dir = repo_url.split("/")[-1].replace(".git", "")
repo_path = os.path.join(base_path, repo_dir)

try:
    subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    print(f"Directory '{repo_dir}' exists. Pulled latest changes...")
except (subprocess.CalledProcessError, FileNotFoundError):
    print(f"Directory '{repo_dir}' not found or invalid. Cloning...")
    subprocess.run(["git", "clone", repo_url], check=True)

os.chdir(repo_path)

# Phase 1: Train MLPs to "grok" basic mathematical operations

**Grokking** = delayed generalization.  
We train well past overfitting (10k-20k epochs each) using AdamW with **heavy weight decay** (the key ingredient for grokking), and the network eventually restructures its weights into an algorithmic circuit that generalizes perfectly.

### Tasks
We train 5 MLPs on the following functions:
- `cos(x)`
- `sin(x)`
- `identity(x)`
- `x + y`
- `x * y`

## Observation
- Training loss drops fast, validation loss later implying grokking (![Grokking Curves: Train vs Validation Loss](paper/grokking_curves.png)).

## Outputs
- [checkpoints/mlp_*.pt](https://huggingface.co/pandevim/cs810/tree/main/phase1/checkpoints)
- [paper/grokking_curves.png](https://huggingface.co/pandevim/cs810/blob/main/phase1/plots/grokking_curves.png)

In [ ]:
%run phase1_train_mlp.py

# Phase 2: Generate Synthetic ODE Data for Testing

We use a **damped harmonic oscillator**:

$$
\ddot{x} = -k \cdot x - c \cdot \dot{x}
$$

i.e: `ẍ = -1.0·x - 0.1·ẋ` (600 time points, small Gaussian noise added)

### First-Order System Form

Rewriting as a system of first-order differential equations:

$$
\frac{dx}{dt} = v
$$

$$
\frac{dv}{dt} = -k \cdot x - c \cdot v
$$

### Key Insight

This system involves **linear combinations of state variables**, which is exactly what our MLP library should be able to discover.

## Outputs
- Time series `[t, x, v, dx/dt, dv/dt]`: [data/oscillator_data.npz](https://huggingface.co/pandevim/cs810/blob/main/phase1/data/oscillator_data.npz)
- Phase portrait and time series: [paper/oscillator_data.png](https://huggingface.co/pandevim/cs810/blob/main/phase1/plots/oscillator_data.png)

In [13]:
%run phase2_generate_data.py

# Phase 3: Neural SINDy — Using Grokked MLPs as Basis Functions

**Standard SINDy**:

$$
\dot{X} = \Theta(X) \cdot \xi
$$

- Where:  
  $$
  \Theta = [1, x, x^2, \sin(x), \cos(x), \dots]
  $$  
  (hand-crafted library)  
  $\xi$ is a sparse coefficient vector

**Neural SINDy**:

$$
\dot{X} = \Theta_{\text{neural}}(X) \cdot \xi
$$

- Where:  
  $$ \Theta_{\text{neural}} = [\{\text{MLP}_{\text{identity}}\}(x), \{\text{MLP}_{\text{cos}}\}(x), \{\text{MLP}_{\text{sin}}\}(x), \dots] $$
  (grokked MLP library)  
  $\xi$ is a sparse coefficient vector found via STLSQ

**Key Insight:**  
Because each MLP is a differentiable function learned to approximate a clean mathematical operation, the sparse regression selects which *operations* best explain the dynamics.

In [14]:
%run phase3_neural_sindy.py

# Phase 4: Run the Full Neural SINDy Discovery Experiment

### Pipeline

1. **Load Grokked MLPs from Checkpoints**
2. **Load ODE Data**
3. **Build Library Matrix Θ** by evaluating each MLP on state data
4. **Solve Sparse Regression**
    - Fit the dynamics using sparse regression: $$ \dot{X} \approx \Theta(X) \cdot \xi $$
    - Determine which MLPs contribute to the dynamics and their coefficients **ξ**.
      * Tries multiple sparsity thresholds, picks the sparsest model with low error
5. **Report Selected MLPs and Coefficients**
6. **Validate Discovered System**
   - Simulate the system using the learned ξ coefficients.
   - Compare the simulation against the ground-truth ODE trajectories to assess accuracy.

### Outputs
- [data/discovery_results.npz](https://huggingface.co/pandevim/cs810/blob/main/phase1/data/discovery_results.npz)
- [paper/discovery_comparison.png](https://huggingface.co/pandevim/cs810/blob/main/phase1/plots/discovery_comparison.png)
- [paper/coefficients.png](https://huggingface.co/pandevim/cs810/blob/main/phase1/plots/coefficients.png)

In [15]:
%run phase4_full_experiment.py

```txt
============================================================
  NEURAL SINDy DISCOVERY
============================================================

  Building library matrix Θ...
  Θ shape: (600, 8) (8 library terms)
  Library terms: ['identity(x)', 'identity(v)', 'cos(x)', 'cos(v)', 'sin(x)', 'sin(v)', 'add(x,v)', 'mul(x,v)']

  ─── Discovering equation for ẋ (dx/dt) ───
  ẋ (dx/dt) = -0.3172·identity(x) +0.6502·identity(v) -0.0123·sin(x) +0.0212·sin(v) +0.3291·add(x,v)

  ─── Discovering equation for v̇ (dv/dt) ───
  v̇ (dv/dt) = -0.6208·identity(x) +0.2621·identity(v) -0.0175·sin(x) -0.3621·add(x,v)

  Threshold=0.010: MSE=0.000001, Active terms=9/16

  Building library matrix Θ...
  Θ shape: (600, 8) (8 library terms)
  Library terms: ['identity(x)', 'identity(v)', 'cos(x)', 'cos(v)', 'sin(x)', 'sin(v)', 'add(x,v)', 'mul(x,v)']

  ─── Discovering equation for ẋ (dx/dt) ───
  ẋ (dx/dt) = -0.3325·identity(x) +0.6675·identity(v) +0.3324·add(x,v)

  ─── Discovering equation for v̇ (dv/dt) ───
  v̇ (dv/dt) = -0.6322·identity(x) +0.2677·identity(v) -0.3676·add(x,v)

  Threshold=0.025: MSE=0.000001, Active terms=6/16

  Building library matrix Θ...
  Θ shape: (600, 8) (8 library terms)
  Library terms: ['identity(x)', 'identity(v)', 'cos(x)', 'cos(v)', 'sin(x)', 'sin(v)', 'add(x,v)', 'mul(x,v)']

  ─── Discovering equation for ẋ (dx/dt) ───
  ẋ (dx/dt) = -0.3325·identity(x) +0.6675·identity(v) +0.3324·add(x,v)

  ─── Discovering equation for v̇ (dv/dt) ───
  v̇ (dv/dt) = -0.6322·identity(x) +0.2677·identity(v) -0.3676·add(x,v)

  Threshold=0.050: MSE=0.000001, Active terms=6/16

  Building library matrix Θ...
  Θ shape: (600, 8) (8 library terms)
  Library terms: ['identity(x)', 'identity(v)', 'cos(x)', 'cos(v)', 'sin(x)', 'sin(v)', 'add(x,v)', 'mul(x,v)']

  ─── Discovering equation for ẋ (dx/dt) ───
  ẋ (dx/dt) = -0.3325·identity(x) +0.6675·identity(v) +0.3324·add(x,v)

  ─── Discovering equation for v̇ (dv/dt) ───
  v̇ (dv/dt) = -0.6322·identity(x) +0.2677·identity(v) -0.3676·add(x,v)

  Threshold=0.100: MSE=0.000001, Active terms=6/16

  Building library matrix Θ...
  Θ shape: (600, 8) (8 library terms)
  Library terms: ['identity(x)', 'identity(v)', 'cos(x)', 'cos(v)', 'sin(x)', 'sin(v)', 'add(x,v)', 'mul(x,v)']

  ─── Discovering equation for ẋ (dx/dt) ───
  ẋ (dx/dt) = -0.3325·identity(x) +0.6675·identity(v) +0.3324·add(x,v)

  ─── Discovering equation for v̇ (dv/dt) ───
  v̇ (dv/dt) = -0.6322·identity(x) +0.2677·identity(v) -0.3676·add(x,v)

  Threshold=0.200: MSE=0.000001, Active terms=6/16

  ✓ Best threshold: 0.2, MSE: 0.000001

============================================================
  DISCOVERED EQUATIONS
============================================================

  ẋ = dx/dt = -0.3325 · identity(x) +0.6675 · identity(v) +0.3324 · add(x,v)

  v̇ = dv/dt = -0.6322 · identity(x) +0.2677 · identity(v) -0.3676 · add(x,v)

  TRUE EQUATIONS:
  ẋ = +1.0 · v
  v̇ = -1.0 · x  -0.1 · v
```

# Analysis from Phase 4
**Result: ✅ Mathematically correct, but coefficients are spread across redundant terms.**

Since `add(x,v) ≈ x + v`, substituting and simplifying:

| Equation | Discovered (simplified) | True | Match? |
|----------|------------------------|------|--------|
| ẋ | (-0.3325 + 0.3324)·x + (0.6675 + 0.3324)·v = **-0.0001·x + 0.9999·v** | +1.0·v | ✅ |
| v̇ | (-0.6322 - 0.3676)·x + (0.2677 - 0.3676)·v = **-0.9998·x - 0.0999·v** | -1.0·x - 0.1·v | ✅ |

**MSE:** 0.000001 (essentially exact fit)  
**Active terms:** 6/16

### Key Finding: Collinearity Problem

The $\text{MLP}_{\text{add}}(x, v) = x + v$ is mathematically identical to $\text{MLP}_{\text{identity}}(x) + \text{MLP}_{\text{identity}}(v)$, creating a rank-deficient library matrix as it makes the column $\text{MLP}_{\text{add}}(x, v)$ in $\Theta(X)$ lie in the span of two other columns $\Theta^T \Theta$. STLSQ (even with ridge α=0.01) cannot distinguish between them, so it distributes the coefficients arbitrarily across the three redundant columns. The *combined* answer is correct, but the individual coefficients are uninterpretable. $\text{MLP}_{\text{mul}}(x, v) = x \cdot v$ is genuinely nonlinear, so it is not redundant and should stay.

### Implication
The library design must avoid including terms that are linear combinations of other terms. For the next run, removing $\text{MLP}_{\text{add}}$ from the library (or adding a collinearity detection step) should produce clean, interpretable coefficients.

### What to change
1. **Primary fix (In Phase 3)**: drop $\text{MLP}_{\text{add}}$ from `binary_names` in `build_default_library`. This removes the redundancy at the source and the discovered coefficients will land on $\text{MLP}_{\text{identity}}(x)$ and $\text{MLP}_{\text{identity}}(v)$ directly, clean and interpretable.

2. **Secondary safeguard (optional):**  
Before STLSQ, run a collinearity check on $\Theta(X)$ so the issue surfaces automatically if new redundant basis functions are added later. Two cheap options:
    - **Rank / condition-number check:** If $\mathrm{cond}(\Theta) > 10^8$ or $\mathrm{rank}(\Theta) < n_{\text{terms}}$, raise a warning and drop the offending column (e.g., the column with the highest pairwise $|\mathrm{corr}|$ with another column, or aligned with the smallest singular direction).
    - **Pairwise correlation prune:** Drop any column with $|\mathrm{corr}| > 0.999$ relative to a lower-indexed column.

### Revelation
**Gumbel-Softmax** sidesteps the root cause. STLSQ fails on rank-deficient $\Theta(X)$ because it solves a linear system. A *Gumbel-Softmax router* picks terms via a differentiable argmax over a learned categorical, there is no _normal-equations inversion_, so rank deficiency doesn't produce the "spread across redundant columns" pathology.


# Phase 5: The Gumbel-Softmax Router experiment

This is the end-to-end differentiable variant of Neural SINDy.
Compare results with Phase 4 (STLSQ baseline).

### Pipeline:
1. Load grokked MLPs (frozen)
2. Load ODE data
3. Train a router network with Gumbel-Softmax to learn which MLP
   explains each component of the dynamics
4. Anneal temperature: (τ=5.0, soft selection, exploratory) → (τ=0.05, hard selection, discrete)
  - Uses Straight-Through Estimator: forward pass routes through 1 MLP, backward pass gets smooth gradients
5. Report which MLPs were selected and their learned coefficients
6. Compare against STLSQ baseline results

### Outputs
- [paper/router_results.png](https://huggingface.co/pandevim/cs810/blob/main/phase1/plots/router_results.png)

**What to look for:**

- Training loss should decrease smoothly as temperature anneals
- Final routing should converge to one-hot (one MLP per derivative)
- Should select the same MLPs as STLSQ with similar coefficients

In [ ]:
from phase5_gumbel_softmax_router import exp1

router1, history1, summary1, scorecard1 = exp1.run(
    load_grokked_mlp=load_grokked_mlp,
    hf_repo_id=HF_REPO_ID,
    device=device,
    upload=True,
)

```txt
============================================================
  ROUTING RESULTS
============================================================

  dx/dt:
        identity(v): selected 43.8% of the time, coefficient = +1.0000
             cos(v): selected 17.5% of the time, coefficient = -0.3764
             cos(x): selected 11.5% of the time, coefficient = -0.7670
           add(x,v): selected 10.2% of the time, coefficient = +0.2997
           mul(x,v): selected 6.7% of the time, coefficient = -0.7885
        identity(x): selected 5.3% of the time, coefficient = -0.4449
             sin(x): selected 5.0% of the time, coefficient = -0.1939

  dv/dt:
        identity(x): selected 19.2% of the time, coefficient = -0.9940
             cos(x): selected 18.2% of the time, coefficient = +0.3412
           add(x,v): selected 14.2% of the time, coefficient = -0.4907
             sin(x): selected 11.2% of the time, coefficient = -0.8817
        identity(v): selected 9.7% of the time, coefficient = +0.2588
             sin(v): selected 9.7% of the time, coefficient = -0.3183
             cos(v): selected 9.2% of the time, coefficient = +0.7209
           mul(x,v): selected 8.8% of the time, coefficient = +0.0448

============================================================
  DISCOVERED EQUATIONS (Gumbel-Softmax)
============================================================

  ẋ = dx/dt = +1.0000 · identity(v) -0.3764 · cos(v) -0.7670 · cos(x) +0.2997 · add(x,v) -0.7885 · mul(x,v) -0.4449 · identity(x) -0.1939 · sin(x)

  v̇ = dv/dt = -0.9940 · identity(x) +0.3412 · cos(x) -0.4907 · add(x,v) -0.8817 · sin(x) +0.2588 · identity(v) -0.3183 · sin(v) +0.7209 · cos(v) +0.0448 · mul(x,v)

  TRUE EQUATIONS:
  ẋ = +1.0 · v
  v̇ = -1.0 · x  -0.1 · v

============================================================
  COMPARISON: STLSQ vs GUMBEL-SOFTMAX
============================================================

  STLSQ discovered:
    ẋ = dx/dt = -0.3325·identity(x) +0.6675·identity(v) +0.3324·add(x,v)
    v̇ = dv/dt = -0.6322·identity(x) +0.2677·identity(v) -0.3676·add(x,v)

  Gumbel-Softmax discovered:
    ẋ = dx/dt = +1.0000·identity(v) -0.3764·cos(v) -0.7670·cos(x) +0.2997·add(x,v) -0.7885·mul(x,v) -0.4449·identity(x) -0.1939·sin(x)
    v̇ = dv/dt = -0.9940·identity(x) +0.3412·cos(x) -0.4907·add(x,v) -0.8817·sin(x) +0.2588·identity(v) -0.3183·sin(v) +0.7209·cos(v) +0.0448·mul(x,v)

  TRUE: ẋ = +1.0·v, v̇ = -1.0·x -0.1·v
```

# Analysis from Phase 5: the router did not commit
The headline failure is visible in the bar chart and the routing percentages: no MLP is selected more than 43.8% of the time for any derivative. A working Gumbel-Softmax run should end with near-one-hot activations (>95% on one term per derivative). Yours looks like a softmax soup.

## What actually worked
- dx/dt → identity(v) with coefficient +1.0000 at 43.8% activation. That coefficient is exact. The router has found the right term, it just hasn't fully committed.
- dv/dt → identity(x) with coefficient -0.9940 at 19.2% activation. Also structurally correct (true value -1.0).

## What broke
1. The damping term is gone. True v̇ has -0.1·v. Your discovered v̇ has +0.2588·identity(v) at 9.7% — wrong sign, wrong magnitude. Damping is 10x smaller than the restoring force, and without a sparsity prior it gets drowned by noise spread across 7 other terms.

2. The soft-mixture escape hatch. With high τ, the router computes a weighted average over all MLPs and learns per-MLP coefficients that combine to fit the data. Nothing in the loss pushes the logits apart. At τ=0.05 you sample discretely but the underlying distribution never concentrated, so get_routing_summary at τ=0.01 still shows 8 terms with non-trivial mass.

3. The collinearity problem is still here — just wearing a new costume. STLSQ split coefficients between identity(x/v) and add(x,v). The router is doing the same thing probabilistically: add(x,v) is selected 10.2% / 14.2% of the time with meaningful coefficients. The earlier claim that "Gumbel-Softmax sidesteps rank deficiency" was mechanically correct but incomplete — rank deficiency became probability-mass splitting.

4. The discontinuity at epoch ~1800 in the loss plot. Train loss jumps from ~negative values (which itself is suspicious — likely a logging artifact of soft-selection masking) to a higher plateau the moment τ hits its floor. That jump is the moment the soft mixture stops hiding the lack of commitment. A well-trained router should show no such discontinuity — it means the soft and hard regimes disagree.

5. Final MSE (~5e-3) is ~1000x worse than STLSQ's 1e-6. Even though STLSQ's coefficients were "spread," its predictions were essentially exact. The router's predictions are genuinely worse, not just less interpretable.

## Verdict
- Experiment 1 (STLSQ): correct answer, ugly presentation.
- Experiment 3 (this run): ugly presentation and wrong answer (missing damping, wrong sign on identity(v) in v̇).

> The router has the capacity to find the right answer — it nailed +1.0·v — but the training objective lets it settle for a soft mixture.

## What I'd try next, in priority order
1. Entropy regularization on the routing logits. Add λ · H(π) with negative sign to the loss (i.e., penalize high entropy). Without this, nothing pushes the router toward one-hot. This alone is likely to fix most of what you see.
2. Longer soak at low τ. Currently τ anneals 5→0.05 over epochs 0-2000, then stays at 0.05 for 1000 epochs. Try extending the low-τ phase to 3000+ epochs, or going to τ=0.01.
3. Drop add(x,v) from the library (the one-line fix from Experiment 2). This removes the probability-mass-splitting attractor and gives the router a cleaner landscape.
4. Sanity check the coefficient parametrization. If the per-MLP coefficient is learned independently of routing probability, the router can minimize loss by picking large coefficients on rarely-selected MLPs. Coupling them (e.g., expected-value loss) may help.
5. Only then worry about the damping term — steps 1-3 may recover it automatically once the router commits.

**Initial Conclusion:** The core issue is missing sparsity pressure, not the method. Gumbel-Softmax without entropy regularization is just a fancy softmax.


# Phase 5: The Gumbel-Softmax Router experiment — Experiment 2

Second pass at the end-to-end differentiable router after Experiment 1 failed
to commit (no MLP selected more than ~44% of the time, damping term lost).

### What changed since Experiment 1

1. **State-independent routing.** The router is no longer a state-dependent
   MLP (`Linear → ReLU → Linear → ReLU → Linear`). It is now a single
   learnable logit vector per derivative, broadcast across the batch.
   - Matches the SINDy assumption that a single term governs the dynamics
     everywhere in state space, not region-by-region.
   - Param count drops from ~9,760 → 16 (2 derivatives × 8 MLPs).
   - Each batch sample still gets its own Gumbel noise draw, so exploration
     is preserved.
2. **Commitment entropy penalty (correct sign, scheduled weight).** The
   previous run *subtracted* entropy from the loss (pushing distributions
   toward *higher* entropy — the wrong direction). Now we *add* a
   temperature-scheduled entropy penalty:
entropy_weight = 0.05 · max(0, 1 − τ / τ_start)
loss += entropy_weight · H(softmax(logits))


- At τ = 5.0 (start): weight = 0 → pure exploration via Gumbel noise.
- At τ = 0.05 (end): weight ≈ 0.0495 → strong pressure toward one-hot.

**What to look for (vs. Experiment 1):**

- `identity(v)` activation on `dx/dt` should jump from 43.8% → ~100%.
- `identity(x)` activation on `dv/dt` should similarly concentrate toward 100%.
- Loss curve should be smooth through the τ-anneal handoff (no jump at
epoch ~1800 like last time).
- Open question: the damping term (`-0.1·v` in `v̇`) is ~10× smaller than
the restoring force. One-hot commitment per derivative may force the
router to miss it. If it does, next step is a top-k variant that allows
2 active terms per derivative.

In [ ]:
from phase5_gumbel_softmax_router import exp2

router2, history2, summary2, scorecard2 = exp2.run(
    load_grokked_mlp=load_grokked_mlp,
    hf_repo_id=HF_REPO_ID,
    device=device,
    upload=True,
)

```txt
============================================================
  ROUTING RESULTS
============================================================

  dx/dt:
             sin(v): selected 99.7% of the time, coefficient = +1.0265

  dv/dt:
             sin(x): selected 99.2% of the time, coefficient = -1.0138

============================================================
  DISCOVERED EQUATIONS (Gumbel-Softmax)
============================================================

  ẋ = dx/dt = +1.0265 · sin(v)

  v̇ = dv/dt = -1.0138 · sin(x)

  TRUE EQUATIONS:
  ẋ = +1.0 · v
  v̇ = -1.0 · x  -0.1 · v

============================================================
  COMPARISON: STLSQ vs GUMBEL-SOFTMAX
============================================================

  STLSQ discovered:
    ẋ = dx/dt = -0.3325·identity(x) +0.6675·identity(v) +0.3324·add(x,v)
    v̇ = dv/dt = -0.6322·identity(x) +0.2677·identity(v) -0.3676·add(x,v)

  Gumbel-Softmax discovered:
    ẋ = dx/dt = +1.0265·sin(v)
    v̇ = dv/dt = -1.0138·sin(x)

  TRUE: ẋ = +1.0·v, v̇ = -1.0·x -0.1·v
```

## Analysis: the router committed cleanly — but to the wrong term

Experiment 2 fixes the problem Experiment 1 failed on (commitment), cleanly
demonstrates a new problem, and confirms a suspicion about the library.

### What worked

- **One-hot routing achieved.** `sin(v)` selected 99.7% of the time for
  `dx/dt`, `sin(x)` selected 99.2% for `dv/dt`. No mass leaking to other
  terms. The entropy penalty + state-independent router did exactly what
  they were designed to do.
- **Smooth loss curve.** The discontinuity at epoch ~1800 from Experiment 1
  is gone. Train loss descends monotonically through the τ handoff, which
  means the soft and hard regimes now agree — the router's logits were
  concentrated *before* the Gumbel samples became effectively deterministic.
- **Router parameter count dropped from 9,760 → 32** (16 logits + 16
  coefficients). The state-independent router is ~300x smaller and generalizes
  better: val loss ~1.2e-3, ~4x better than Experiment 1's ~5e-3.
- **Coefficients are near the true magnitudes** (+1.0265 and -1.0138 vs.
  true ±1.0). That's the right ballpark for the restoring term.

### What broke

1. **The router chose `sin` over `identity`.** True equations are linear in
   `x` and `v`, so `identity(x)` and `identity(v)` should have won. The
   router picked `sin(x)` and `sin(v)` instead. This is a **library
   degeneracy** — for the amplitudes the oscillator visits (|x|, |v| ≲ 1),
   `sin(z) ≈ z` to within ~15% at the peaks. Both basis functions are
   *approximately* collinear, and the entropy penalty broke the tie
   arbitrarily.

   The coefficient inflation (+1.0265 vs. true +1.0) is the tell: since
   `|sin(z)| < |z|`, the coefficient has to absorb the Taylor-series
   shortfall. Had the router picked identity, the coefficient would have
   landed exactly on 1.0.

2. **Damping is still gone.** True `v̇ = -1.0·x - 0.1·v`. Discovered
   `v̇ = -1.0138·sin(x)`. The `-0.1·v` term is ~10x smaller than the
   restoring force and one-hot commitment *cannot* represent a two-term
   equation. This is a structural limitation of the current router, not a
   training failure.

3. **MSE is ~5000x worse than STLSQ.** Final train MSE ~5.6e-3 vs. STLSQ's
   ~1e-6. The sin-vs-identity approximation error dominates. Gumbel-Softmax
   is now more interpretable than STLSQ (two clean one-hot selections vs.
   six-term spread) but less accurate.

4. **The val-loss spike at τ=0.05 (epoch 2000).** A one-epoch jump from
   ~1.2e-3 to ~3.5e-3 when τ hits its floor. This is the residual of
   Experiment 1's problem in miniature — the router's top-1 choice wasn't
   *quite* stable yet when hard sampling became deterministic. It recovers
   within 200 epochs, so it's benign, but a slower anneal would smooth it.

### The deeper insight

**The library has two near-collinearities:**
- *Exact* collinearity: `add(x,v) = identity(x) + identity(v)` — broke STLSQ.
- *Approximate* collinearity: `sin(z) ≈ identity(z)` at small amplitudes —
  broke the one-hot router.

Experiment 1 (STLSQ) hit the first. Experiment 2 (Gumbel-Softmax) sidesteps
the first but hits the second. Neither method is "failing" — they're both
revealing that **the library design is the real bottleneck.** A library with
redundant terms will always put the burden on the optimizer to disambiguate,
and different optimizers fail in different ways.

### What I'd try next, in priority order

1. **Allow top-k routing per derivative**. The router must be allowed to sample 2 or 3 terms simultaneously (e.g., using multiple Gumbel-Softmax vectors or a continuous relaxation of k-hot categorical distributions) to capture multi-term equations like damping. One `sin` term cannot represent `v̇ = -1·x - 0.1·v`. A two-term selection can.  
**Implementation:** pick top-2 logits per derivative instead of argmax, pass both through with their learned coefficients. This alone should recover damping.

2. **Prefer simpler basis via a complexity prior.** Add a per-term
   "complexity penalty" in the logits — e.g., `identity` gets a log-prior
   bonus of +α over `sin`/`cos`. Breaks the sin/identity tie in favor of
   the simpler (and correct) term. This is a cheap, honest fix: Occam's
   razor baked into the router.

3. **Test the hypothesis directly.** Re-run Experiment 2 with `sin` and
   `cos` *removed* from the library (leaving only `identity`, `add`, `mul`).
   If the router then picks `identity(v)` / `identity(x)` with coefficients
   converging to ±1.0, that confirms the sin-vs-identity degeneracy is the
   sole reason for the wrong selection.

4. **Only then revisit the damping question.** If 1 and 2 don't recover
   `-0.1·v`, the issue is data (600 points may not be enough signal for a
   term 10× smaller than the dominant one) rather than routing.

### Scorecard vs. Experiment 1

| Metric                       | Exp 1 (MLP router)      | Exp 2 (scalar router)  |
|------------------------------|-------------------------|------------------------|
| Router params                | 9,760                   | 32                     |
| Max activation (dx/dt)       | 43.8%                   | 99.7%                  |
| Max activation (dv/dt)       | 19.2%                   | 99.2%                  |
| Val MSE                      | ~5e-3                   | ~1.2e-3                |
| Loss discontinuity at τ drop | Large                   | Small spike, recovers  |
| Equations interpretable?     | No (7-8 terms each)     | Yes (1 term each)      |
| Equations correct?           | Structurally yes        | Wrong basis, damping gone |

Experiment 2 is not yet the right answer, but it is a **cleaner failure**
than Experiment 1 — and a cleaner failure is a better foundation for the
next iteration. We've gone from "router can't commit" to "router commits
confidently to a near-correct but simpler-is-better answer." That's
progress.

# Phase 5 — Experiment 3: Top-k Gumbel-Softmax Router with a Complexity Prior.

Motivation (from Experiment 2's analysis):
  - Exp 2 achieved clean one-hot commitment but landed on the wrong basis:
    picked sin(x) / sin(v) instead of identity(x) / identity(v), because
    sin(z) ≈ z for |z| ≲ 1 (approximate collinearity).
  - Exp 2 also cannot represent multi-term equations. True v̇ = -1.0·x - 0.1·v
    needs TWO active basis functions; one-hot routing structurally cannot
    express it.

What changes in Experiment 3:
  1. Top-k routing (k=2). Each derivative selects k basis functions instead
     of 1. Implemented as k independent Gumbel-Softmax draws per derivative
     with the already-selected logits masked out between draws (sampling-
     without-replacement via iterative masked argmax + STE gradients).
     Each slot has its own learnable coefficient. This makes
     "v̇ = -1·x - 0.1·v" representable.

  2. Complexity prior on the logits. Occam's razor baked in: identity is
     cheaper than sin/cos which are cheaper than the binary MLPs. Prior is
     added directly to the routing logits so it biases selection without
     distorting coefficients.

  3. Entropy penalty kept, temperature anneal kept. Both are carried over
     from Exp 2 unchanged; they remain the mechanism for commitment.

  4. Slower low-τ anneal floor and a longer soak at the floor, to smooth
     the small val-loss spike Exp 2 showed when τ hit 0.05.

Expected outcome:
  - dx/dt → identity(v) at ~1.0 (plus a small residual 2nd slot).
  - dv/dt → identity(x) at ~-1.0 AND identity(v) at ~-0.1. Damping recovered.
  - Val MSE should approach STLSQ's ~1e-6, not Exp 2's ~1.2e-3.

This file is self-contained — it re-imports torch/numpy and redefines the
gumbel_softmax helper and build_router_from_checkpoints so it can be dropped
into the notebook without touching the Exp 2 cells. At paste time, you'll
remove the duplicate helper/imports and keep only what Exp 3 adds.

In [ ]:
from phase5_gumbel_softmax_router import exp3

router3, history3, summary3, scorecard3 = exp3.run(
    load_grokked_mlp=load_grokked_mlp,
    hf_repo_id=HF_REPO_ID,
    device=device,
    k=2, alpha=1.0,
    upload=True,
)

In [ ]:
import pandas as pd

# Compare all three experiments side-by-side
scorecards = pd.DataFrame([scorecard1, scorecard2, scorecard3])
scorecards = scorecards.set_index("exp_id")
print(scorecards[["n_params", "val_mse", "commitment", "correctness", "final_tau"]].to_string())
print()
for _, row in scorecards.iterrows():
    print(f"{row.name}:")
    for deriv, eq in row["equations"].items():
        print(f"  {deriv}: {eq}")